# Search-3-Informed (C#) : Recherche Informée

**Navigation** : [<< Recherche non informée (Search-2 C#)](Search-2-Uninformed-Csharp.ipynb) | [Index série Search](../README.md) | [Recherche locale (Search-4 C#) >>](Search-4-LocalSearch-Csharp.ipynb)

**Jumeau .NET** du notebook Python [Search-3-Informed](Search-3-Informed.ipynb). Ce notebook est le port C# / .NET 9.0 des trois algorithmes informés fondamentaux : **Greedy Best-First**, **A\*** et **IDA\***, illustrés sur le graphe des villes françaises (cf Search-2-Csharp) puis sur le **8-puzzle** pour les notions d'admissibilité et de consistance des heuristiques.

## Pourquoi un jumeau C# ?

La recherche informée exploite une connaissance du problème — une **heuristique** $h(n)$ qui estime le coût restant vers le but — pour guider l'exploration vers les régions prometteuses de l'espace. Ce notebook montre, en C#, comment chaque algorithme combine différemment le coût déjà accumulé $g(n)$ et l'estimation $h(n)$ :

- **Greedy Best-First** : fonce vers le but en minimisant $h(n)$ seul — rapide mais **sous-optimal**.
- **A\*** : équilibre $f(n) = g(n) + h(n)$ — **optimal** si $h$ est admissible.
- **IDA\*** : A* à approfondissement itéré par borne sur $f$ — optimal et **borné en mémoire** $O(bd)$.

### Objectifs d'apprentissage

1. **Distinguer** recherche informée et non informée (rôle de l'heuristique).
2. **Implémenter** Greedy, A* et IDA* en C# avec `PriorityQueue<TElement, TPriority>`.
3. **Concevoir** des heuristiques **admissibles** et **consistantes** (8-puzzle).
4. **Comparer** expérimentalement Greedy vs A* et mettre en évidence la **divergence** sur un exemple concret.

### Prérequis

- Notebook [Search-2-Uninformed (C#)](Search-2-Uninformed-Csharp.ipynb) (BFS, DFS, UCS, notion de frontière).
- Bases de C# : `Dictionary`, `PriorityQueue`, délégués `Func<,>`.

**Durée estimée : 45 minutes**

> **Fidélité .NET ⇄ Python (G.9).** Les heuristiques à vol d'oiseau (Haversine sur lat/lon) sont déterministes et identiques entre les deux jumeaux. En revanche les **structures de données** (`PriorityQueue` BCL vs `heapq`) et l'ordre de parcours des voisins d'un `Dictionary` peuvent altérer l'ordre d'expansion exact et le décompte des nœuds — mais les **propriétés pédagogiques** (Greedy sous-optimal, A* optimal, IDA* mémoire linéaire) sont intégralement préservées. C'est l'enseignement visé.

---
## 1. Cadre : problème de recherche et heuristique

### Le 5-uplet (S, A, T, G, c)

Tout problème de recherche se définit par :

| Symbole | Rôle |
|---------|------|
| $S$ | État initial |
| $A(s)$ | Actions possibles depuis l'état $s$ |
| $T(s,a)$ | Successeur (transition) |
| $G$ | Test du but |
| $c(s,a,s')$ | Coût d'une action |

La recherche **non informée** (Search-2) n'utilise que ces cinq éléments. La recherche **informée** y ajoute une **heuristique** :

$$h(n) \approx \text{coût optimal restant de } n \text{ au but}$$

### Heuristique : définition

Une bonne heuristique $h(n)$ :

- **Estime** le coût restant de $n$ au but (sans le calculer exactement).
- **Est rapide** à calculer (bien moins qu'une recherche complète).
- **Est admissible** : $h(n) \leq h^*(n)$ — ne surestime jamais le coût réel $h^*(n)$.

> **Mnémotechnique** : « heuristique » vient du grec *heuriskein* = « trouver ». C'est une règle pratique, pas une garantie mathématique.

### Exemples canoniques

| Problème | Heuristique $h(n)$ |
|----------|--------------------|
| Itinéraire routier | Distance à vol d'oiseau |
| 8-puzzle | Tuiles mal placées ; distance Manhattan |
| Pathfinding grille | Distance Manhattan / Euclidienne |

> *Ancres savantes — Hart, Nilsson & Raphael (1968) ont formalisé A\* ; Korf (1985) a introduit IDA\* ; Russell & Norvig (2020, ch. 3.5) synthétisent la théorie des heuristiques admissibles/consistantes.*

In [1]:
// Imports + structures communes (Node, Problem, GraphProblem, SearchResult).
// Identiques au jumeau Search-2-Csharp pour garder la serie coherente.
using System;
using System.Collections.Generic;
using System.Diagnostics;
using System.Globalization;
using System.Linq;

// Noeud de recherche : etat + parent + action + cout cumule g(n).
public sealed class Node {
    public string State { get; }
    public Node Parent { get; }
    public string Action { get; }
    public double PathCost { get; }   // g(n)
    public int Depth { get; }

    public Node(string state, Node parent = null, string action = null, double pathCost = 0) {
        State = state; Parent = parent; Action = action; PathCost = pathCost;
        Depth = parent == null ? 0 : parent.Depth + 1;
    }

    // Genere les fils en appliquant toutes les actions validables du probleme.
    public IEnumerable<Node> Expand(Problem problem) {
        foreach (var action in problem.Actions(State)) {
            var nextState = problem.Result(State, action);
            var stepCost = problem.StepCost(State, action, nextState);
            yield return new Node(nextState, this, action, PathCost + stepCost);
        }
    }

    // Chemin racine -> this (top-down).
    public IEnumerable<Node> Path() {
        var stack = new Stack<Node>();
        for (var n = this; n != null; n = n.Parent) stack.Push(n);
        return stack;
    }

    public override string ToString() => State;
}

// Probleme abstrait : un etat initial, un but, des actions, un cout.
public abstract class Problem {
    public string Initial { get; }
    public string Goal { get; }
    protected Problem(string initial, string goal) { Initial = initial; Goal = goal; }
    public abstract IEnumerable<string> Actions(string state);
    public abstract string Result(string state, string action);
    public virtual double StepCost(string state, string action, string nextState) => 1.0;
    public virtual bool GoalTest(string state) => state == Goal;
}

// Probleme de cheminement sur graphe pondere.
public sealed class GraphProblem : Problem {
    public Dictionary<string, Dictionary<string, double>> Graph { get; }
    public GraphProblem(string initial, string goal,
                        Dictionary<string, Dictionary<string, double>> graph) : base(initial, goal) {
        Graph = graph;
    }
    public override IEnumerable<string> Actions(string state) {
        if (Graph.TryGetValue(state, out var neighbors)) return neighbors.Keys;
        return Enumerable.Empty<string>();
    }
    public override string Result(string state, string action) => action;
    public override double StepCost(string state, string action, string nextState) {
        if (Graph.TryGetValue(state, out var neighbors) && neighbors.TryGetValue(nextState, out var c)) return c;
        return double.PositiveInfinity;
    }
}

// Resultat de recherche avec statistiques (cout, noeuds, temps).
public sealed class SearchResult {
    public string Algorithm { get; }
    public Node SolutionNode { get; }
    public bool Found => SolutionNode != null;
    public IEnumerable<string> PathStates => SolutionNode?.Path().Select(n => n.State) ?? Enumerable.Empty<string>();
    public double Cost => SolutionNode?.PathCost ?? double.PositiveInfinity;
    public int NodesExpanded { get; }
    public int NodesGenerated { get; }
    public int MaxFrontierSize { get; }
    public double ElapsedMs { get; }
    public List<string> ExploredOrder { get; }

    public SearchResult(string algo, Node solution, int expanded, int generated,
                        int maxFrontier, double elapsedMs, List<string> exploredOrder) {
        Algorithm = algo; SolutionNode = solution;
        NodesExpanded = expanded; NodesGenerated = generated;
        MaxFrontierSize = maxFrontier; ElapsedMs = elapsedMs;
        ExploredOrder = exploredOrder;
    }

    public void Display() {
        var fr = CultureInfo.GetCultureInfo("fr-FR");
        Console.WriteLine($"--- {Algorithm} ---");
        if (Found) {
            Console.WriteLine($"  Chemin : {string.Join(" -> ", PathStates)}");
            Console.WriteLine($"  Cout   : {Cost.ToString("F0", fr)}");
            Console.WriteLine($"  Sauts  : {SolutionNode.Depth}");
        } else {
            Console.WriteLine("  ECHEC : aucune solution trouvee");
        }
        Console.WriteLine($"  Noeuds explores : {NodesExpanded}");
        Console.WriteLine($"  Noeuds generes  : {NodesGenerated}");
        Console.WriteLine($"  Pic frontiere   : {MaxFrontierSize}");
        Console.WriteLine($"  Temps           : {ElapsedMs.ToString("F2", fr)} ms");
    }
}

Console.WriteLine("Framework de recherche informee pret (Node, Problem, GraphProblem, SearchResult).");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Framework de recherche informee pret (Node, Problem, GraphProblem, SearchResult).


---
## 2. Graphe des villes françaises + heuristique à vol d'oiseau

Nous réutilisons le **même graphe pondéré** que le jumeau Search-2-Csharp (13 villes, distances routières en km). Pour la recherche informée, nous ajoutons une **heuristique** $h(n)$ = distance à vol d'oiseau entre la ville courante et le but, calculée par la **formule de Haversine** sur les coordonnées lat/lon.

**Pourquoi Haversine est admissible** : à vol d'oiseau (arc de grand cercle), la distance est toujours **inférieure ou égale** à la distance routière réelle (qui suit des arcs de cercle déformés par le réseau). Donc $h(n) \leq h^*(n)$ pour toute ville — la condition d'admissibilité.

In [2]:
// Graphe des villes francaises (13 villes, reutilise du jumeau Search-2-Csharp)
// + coordonnees lat/lon + heuristique Haversine.
var franceGraph = new Dictionary<string, Dictionary<string, double>> {
    ["Bordeaux"]    = new() { ["Paris"] = 585, ["Toulouse"] = 245, ["Nantes"] = 350 },
    ["Paris"]       = new() { ["Bordeaux"] = 585, ["Lille"] = 225, ["Strasbourg"] = 490, ["Lyon"] = 465, ["Nantes"] = 385 },
    ["Lille"]       = new() { ["Paris"] = 225, ["Rennes"] = 510 },
    ["Rennes"]      = new() { ["Lille"] = 510, ["Paris"] = 385, ["Nantes"] = 110, ["Brest"] = 245 },
    ["Brest"]       = new() { ["Rennes"] = 245 },
    ["Nantes"]      = new() { ["Rennes"] = 110, ["Paris"] = 385, ["Bordeaux"] = 350, ["Toulouse"] = 590 },
    ["Strasbourg"]  = new() { ["Paris"] = 490, ["Lyon"] = 490 },
    ["Lyon"]        = new() { ["Paris"] = 465, ["Strasbourg"] = 490, ["Marseille"] = 315, ["Toulouse"] = 535, ["Grenoble"] = 115 },
    ["Grenoble"]    = new() { ["Lyon"] = 115, ["Marseille"] = 305 },
    ["Marseille"]   = new() { ["Lyon"] = 315, ["Grenoble"] = 305, ["Toulouse"] = 405, ["Nice"] = 200 },
    ["Toulouse"]    = new() { ["Bordeaux"] = 245, ["Nantes"] = 590, ["Marseille"] = 405, ["Lyon"] = 535, ["Montpellier"] = 245 },
    ["Montpellier"] = new() { ["Toulouse"] = 245, ["Marseille"] = 165 },
    ["Nice"]        = new() { ["Marseille"] = 200 },
};

// Coordonnees (latitude, longitude) en degres decimaux.
var franceCoords = new Dictionary<string, (double lat, double lon)> {
    ["Paris"]       = (48.86, 2.35),
    ["Lyon"]        = (45.76, 4.83),
    ["Marseille"]   = (43.30, 5.37),
    ["Toulouse"]    = (43.60, 1.44),
    ["Bordeaux"]    = (44.84, -0.57),
    ["Nantes"]      = (47.22, -1.55),
    ["Rennes"]      = (48.11, -1.68),
    ["Lille"]       = (50.63, 3.06),
    ["Strasbourg"]  = (48.57, 7.75),
    ["Nice"]        = (43.70, 7.26),
    ["Montpellier"] = (43.61, 3.88),
    ["Grenoble"]    = (45.19, 5.72),
    ["Brest"]       = (48.39, -4.49),
};

// Distance Haversine (arc de grand cercle) entre deux villes — heuristique admissible.
double Haversine(string a, string b) {
    const double R = 6371.0;  // rayon de la Terre en km
    var (la, lo) = franceCoords[a];
    var (lb, lb2) = franceCoords[b];
    double dLat = (lb - la) * Math.PI / 180.0;
    double dLon = (lb2 - lo) * Math.PI / 180.0;
    double x = Math.Sin(dLat / 2) * Math.Sin(dLat / 2)
             + Math.Cos(la * Math.PI / 180.0) * Math.Cos(lb * Math.PI / 180.0)
             * Math.Sin(dLon / 2) * Math.Sin(dLon / 2);
    return 2.0 * R * Math.Asin(Math.Min(1.0, Math.Sqrt(x)));
}

// Fabrique une fonction heurique h_etat(s) = Haversine(s, goal).
Func<string, double> MakeH(string goal) => s => Haversine(s, goal);

// Probleme de reference : Lille -> Toulouse (la ou Greedy et A* divergent).
var problemLT = new GraphProblem("Lille", "Toulouse", franceGraph);
Func<string, double> hLilleToulouse = MakeH("Toulouse");

Console.WriteLine($"Carte chargee : {franceGraph.Count} villes");
Console.WriteLine($"Probleme de reference : {problemLT.Initial} -> {problemLT.Goal}");
Console.WriteLine();
Console.WriteLine("Heuristique h(n) = Haversine vers Toulouse (vol d'oiseau, admissible) :");
foreach (var v in new[] { "Lille", "Paris", "Rennes", "Nantes", "Bordeaux", "Toulouse" }) {
    Console.WriteLine($"  {v,-12} h = {hLilleToulouse(v),6:F0} km");
}

Carte chargee : 13 villes


Probleme de reference : Lille -> Toulouse


Heuristique h(n) = Haversine vers Toulouse (vol d'oiseau, admissible) :


  Lille        h =    791 km


  Paris        h =    589 km


  Rennes       h =    557 km


  Nantes       h =    465 km


  Bordeaux     h =    211 km


  Toulouse     h =      0 km


### Interprétation : l'heuristique guide vers le but

La colonne $h(n)$ décroît à mesure qu'on se rapproche géographiquement de Toulouse. **Lille** (~830 km à vol d'oiseau) et **Brest** (lointain) ont un $h$ élevé ; **Bordeaux** (~245 km) et surtout **Toulouse** ($h = 0$) ont un $h$ faible. C'est ce gradient que les algorithmes informés vont exploiter.

**Admissibilité** : la distance à vol d'oiseau est toujours $\leq$ à la distance routière (le réseau suit des arcs déformés), donc $h(n) \leq h^*(n)$ partout — condition d'admissibilité vérifiée pour tout but.

---

## 3. Greedy Best-First Search

### Principe

Le **Greedy Best-First** étend toujours le nœud de **plus petite heuristique** $h(n)$ — il « fonce » vers le but en ignorant le coût déjà accumulé $g(n)$.

- **Frontière** : file de priorité ordonnée par $h(n)$ seul.
- **Comportement** : rapide, peu de nœuds explorés.
- **Faiblesse** : **non optimal** et potentiellement **incomplet** (sans explored-set, peut boucler).

| Propriété | Valeur |
|-----------|--------|
| Complétude | Non (sans explored-set) / Oui (avec, graphe fini) |
| Optimalité | **Non** |
| Complexité temps | $O(b^m)$ pire cas |

In [3]:
// Greedy Best-First Search : frontiere par h(n) seul, explored-set anti-boucle.
SearchResult GreedyBestFirst(GraphProblem problem, Func<string, double> h, bool verbose = false) {
    var sw = Stopwatch.StartNew();
    var start = new Node(problem.Initial);
    if (problem.GoalTest(start.State)) {
        sw.Stop();
        return new SearchResult("Greedy", start, 0, 0, 1, sw.Elapsed.TotalMilliseconds, new List<string>());
    }
    // PriorityQueue BCL (.NET 6+) : Dequeue retire le min (ici priorite = h seul).
    // Le tiebreaker (int) garantit un ordre deterministe en cas d'egalite de h.
    var frontier = new PriorityQueue<Node, (double h, int tie)>();
    int counter = 0;
    frontier.Enqueue(start, (h(start.State), counter++));
    var frontierH = new Dictionary<string, double> { [start.State] = h(start.State) };
    var explored = new HashSet<string>();
    var exploredOrder = new List<string>();
    int expanded = 0, generated = 0, maxFrontier = 1;

    while (frontier.Count > 0) {
        var current = frontier.Dequeue();
        if (explored.Contains(current.State)) continue;  // entree stale
        if (problem.GoalTest(current.State)) {
            exploredOrder.Add(current.State);
            sw.Stop();
            return new SearchResult("Greedy", current, expanded, generated, maxFrontier, sw.Elapsed.TotalMilliseconds, exploredOrder);
        }
        explored.Add(current.State);
        exploredOrder.Add(current.State);
        expanded++;
        if (verbose)
            Console.WriteLine($"  Explore: {current.State,-12} g={current.PathCost,6:F0} h={h(current.State),6:F0}");
        foreach (var child in current.Expand(problem)) {
            generated++;
            if (explored.Contains(child.State)) continue;
            double childH = h(child.State);
            if (!frontierH.TryGetValue(child.State, out var prevH) || childH < prevH) {
                frontier.Enqueue(child, (childH, counter++));
                frontierH[child.State] = childH;
                maxFrontier = Math.Max(maxFrontier, frontier.Count);
            }
        }
    }
    sw.Stop();
    return new SearchResult("Greedy", null, expanded, generated, maxFrontier, sw.Elapsed.TotalMilliseconds, exploredOrder);
}

Console.WriteLine("Algorithme Greedy Best-First implemente.");

Algorithme Greedy Best-First implemente.


In [4]:
// Test de Greedy sur Lille -> Toulouse (la ou Greedy et A* divergent).
Console.WriteLine("Greedy Best-First : Lille -> Toulouse");
Console.WriteLine(new string('=', 60));
var resultGreedy = GreedyBestFirst(problemLT, hLilleToulouse, verbose: true);
Console.WriteLine();
resultGreedy.Display();

Greedy Best-First : Lille -> Toulouse


  Explore: Lille        g=     0 h=   791


  Explore: Rennes       g=   510 h=   557


  Explore: Nantes       g=   620 h=   465


--- Greedy ---


...

  Sauts  : 3


  Noeuds explores : 3


  Noeuds generes  : 10


  Pic frontiere   : 4


  Temps           : 8,95 ms


### Interprétation : Greedy fonce vers l'ouest

**Sortie obtenue** : Greedy explore d'abord le voisin de Lille dont le $h$ est le plus faible. Depuis Lille, les deux voisins sont **Paris** ($h \approx 589$ km) et **Rennes** ($h \approx 555$ km) — Rennes est **légèrement plus proche à vol d'oiseau** de Toulouse, donc Greedy part vers l'ouest (Bretagne), puis descend via Nantes. Il finit par emprunter l'arête directe **Nantes → Toulouse** (590 km) parce qu'elle mène au but le plus vite géométriquement.

| Aspect | Observation |
|--------|-------------|
| Critère | Minimise $h(n)$ seul, ignore $g(n)$ |
| Chemin trouvé | Lille → Rennes → Nantes → Toulouse |
| Coût | ~1210 km — **sous-optimal** |
| Sauts | 3 |

**Leçon** : Greedy est **piégé par la géométrie**. Il ne « voit » pas que le chemin via Paris et Bordeaux, bien que géométriquement plus long à chaque saut individuel, est **plus court en coût cumulé**. C'est exactement la faiblesse qu'A* va corriger.

---
## 4. A* Search

### Principe

A* combine le coût passé et l'estimation future dans une **fonction d'évaluation unique** :

$$f(n) = g(n) + h(n)$$

- $g(n)$ : coût exact du chemin depuis l'initial (connu).
- $h(n)$ : estimation du coût restant (heuristique).
- $f(n)$ : estimation du coût total d'un chemin passant par $n$.

A* étend le nœud de **plus petit $f(n)$**, équilibrant **exploitation** du chemin engagé et **exploration** vers le but.

### Propriétés

| Propriété | Valeur | Condition |
|-----------|--------|-----------|
| Complétude | Oui | branchements finis, coûts > 0 |
| Optimalité | **Oui** | si $h$ **admissible** |
| Optimalité | **Oui** | si $h$ **consistante** (et Graph-Search, pas de re-open) |
| Mémoire | $O(b^d)$ | exponentielle |

> **Théorème (Hart, Nilsson, Raphael 1968)** : si $h$ est admissible, A* trouve un chemin de coût minimal.

In [5]:
// A* Search : frontiere par f = g + h, g-score pour ne pas re-expandre moins bien.
SearchResult AStar(GraphProblem problem, Func<string, double> h, bool verbose = false) {
    var sw = Stopwatch.StartNew();
    var start = new Node(problem.Initial);
    if (problem.GoalTest(start.State)) {
        sw.Stop();
        return new SearchResult("A*", start, 0, 0, 1, sw.Elapsed.TotalMilliseconds, new List<string>());
    }
    var frontier = new PriorityQueue<Node, (double f, int tie)>();
    int counter = 0;
    frontier.Enqueue(start, (h(start.State), counter++));   // g(initial) = 0
    var gScore = new Dictionary<string, double> { [start.State] = 0 };
    var explored = new HashSet<string>();
    var exploredOrder = new List<string>();
    int expanded = 0, generated = 0, maxFrontier = 1;

    while (frontier.Count > 0) {
        var current = frontier.Dequeue();
        if (explored.Contains(current.State)) continue;  // cle stale : un meilleur g a deja ete expande
        if (problem.GoalTest(current.State)) {
            exploredOrder.Add(current.State);
            sw.Stop();
            return new SearchResult("A*", current, expanded, generated, maxFrontier, sw.Elapsed.TotalMilliseconds, exploredOrder);
        }
        explored.Add(current.State);
        exploredOrder.Add(current.State);
        expanded++;
        if (verbose)
            Console.WriteLine($"  Explore: {current.State,-12} g={current.PathCost,6:F0} h={h(current.State),6:F0} f={current.PathCost + h(current.State),7:F0}");
        foreach (var child in current.Expand(problem)) {
            generated++;
            if (explored.Contains(child.State)) continue;
            if (!gScore.TryGetValue(child.State, out var prevG) || child.PathCost < prevG) {
                gScore[child.State] = child.PathCost;
                frontier.Enqueue(child, (child.PathCost + h(child.State), counter++));
                maxFrontier = Math.Max(maxFrontier, frontier.Count);
            }
        }
    }
    sw.Stop();
    return new SearchResult("A*", null, expanded, generated, maxFrontier, sw.Elapsed.TotalMilliseconds, exploredOrder);
}

Console.WriteLine("Algorithme A* implemente.");

Algorithme A* implemente.


In [6]:
// Test de A* sur Lille -> Toulouse : doit trouver le chemin optimal ~1055 km.
Console.WriteLine("A* : Lille -> Toulouse");
Console.WriteLine(new string('=', 60));
var resultAStar = AStar(problemLT, hLilleToulouse, verbose: true);
Console.WriteLine();
resultAStar.Display();

A* : Lille -> Toulouse


  Explore: Lille        g=     0 h=   791 f=    791


  Explore: Paris        g=   225 h=   589 f=    814


  Explore: Bordeaux     g=   810 h=   211 f=   1021


  Explore: Lyon         g=   690 h=   360 f=   1050


...

  Sauts  : 3


  Noeuds explores : 4


  Noeuds generes  : 15


  Pic frontiere   : 6


  Temps           : 1,17 ms


### Interprétation : A* équilibre $g$ et $h$

**Sortie obtenue** : A* explore **Paris** avant **Rennes**. Pourquoi ? Parce qu'il tient compte du coût déjà payé :

| Nœud frontière | $g$ | $h$ | $f = g + h$ |
|----------------|-----|-----|-------------|
| Lille → Paris | 225 | 589 | **814** (choisi) |
| Lille → Rennes | 510 | 555 | 1065 |

Bien que Rennes ait un $h$ plus faible (Greedy la préférerait — et l'a préférée), son $g$ est plus du double (510 vs 225). La somme $f$ désigne donc **Paris**, qui mène à la suite optimale **Paris → Bordeaux → Toulouse** = 1055 km.

**C'est la leçon fondamentale** : A* ne se laisse pas berner par un $h$ flatteur. Il paie le coût réel du détour et compare les **coûts totaux estimés**, ce qui garantit l'optimalité sous admissibilité de $h$.

In [7]:
// Comparaison cote a cote : Greedy vs A* sur Lille -> Toulouse.
var fr = CultureInfo.GetCultureInfo("fr-FR");
Console.WriteLine("Comparaison : Greedy vs A* sur Lille -> Toulouse");
Console.WriteLine(new string('=', 70));
Console.WriteLine($"{"Algorithme",-10} {"Chemin",-42} {"Cout",8} {"Sauts",6} {"Noeuds",8}");
Console.WriteLine(new string('-', 70));
foreach (var r in new[] { resultGreedy, resultAStar }) {
    string chemin = string.Join(" -> ", r.PathStates);
    if (chemin.Length > 40) chemin = chemin.Substring(0, 37) + "...";
    Console.WriteLine($"{r.Algorithm,-10} {chemin,-42} {r.Cost.ToString("F0", fr),8} {r.SolutionNode.Depth,6} {r.NodesExpanded,8}");
}
Console.WriteLine();
double diff = resultGreedy.Cost - resultAStar.Cost;
double pct = diff / resultAStar.Cost * 100.0;
if (diff > 0.5) {
    Console.WriteLine($"Greedy a trouve un chemin {diff.ToString("F0", fr)} km plus long ({pct.ToString("F1", fr)}% de surcout).");
    Console.WriteLine("-> Greedy est SOUS-OPTIMAL : il a sacrifie l'optimalite pour explorer moins de noeuds.");
} else {
    Console.WriteLine("Ici Greedy coincide avec A* (pas de divergence sur ce couple).");
}

Comparaison : Greedy vs A* sur Lille -> Toulouse


Algorithme Chemin                                         Cout  Sauts   Noeuds


----------------------------------------------------------------------


Greedy     Lille -> Rennes -> Nantes -> Toulouse          1210      3        3


A*         Lille -> Paris -> Bordeaux -> Toulouse         1055      3        4


Greedy a trouve un chemin 155 km plus long (14,7% de surcout).


-> Greedy est SOUS-OPTIMAL : il a sacrifie l'optimalite pour explorer moins de noeuds.


### Interprétation : la divergence Greedy vs A*

Le tableau met en évidence la **divergence** fondamentale entre les deux stratégies sur le couple **Lille → Toulouse** :

| | Greedy | A* |
|---|--------|-----|
| Chemin | Lille → Rennes → Nantes → Toulouse | Lille → Paris → Bordeaux → Toulouse |
| Coût | ~1210 km | **~1055 km** (optimal) |
| Sauts | 3 | 3 |
| Nœuds explorés | moins | légèrement plus |

**Diagnostic honnête** : Greedy est **plus économe en nœuds explorés** mais **sous-optimal de ~14%** ici. A* paie quelques expansions supplémentaires pour **garantir le chemin minimal**. C'est le compromis exploration vs optimalité — Greedy parie que « la flèche géométrique » est le bon chemin, A* **vérifie**.

> **Remarque de non-trivialité (Prong B, Epic #3801)** : nous avons délibérément choisi un couple START–GOAL où Greedy et A* **divergent** (chemins différents). Sur un graphe à coûts uniformes ou une topologie « en ligne droite », Greedy et A* coïncideraient et la comparaison serait triviale. Le couple Lille → Toulouse, avec sa tension entre la pointe bretonne et la diagonale Paris-Bordeaux, **exerce vraiment** la capacité d'A* à résister à l'attraction géométrique.

---

## 5. IDA* (Iterative Deepening A*)

### Principe

IDA* est l'**approfondissement itéré** appliqué à A* : une suite de **DFS bornés par $f$**, où la borne augmente à chaque itération jusqu'à atteindre le $f$ de la solution optimale.

| Itération | Borne $f_{\text{limit}}$ | Action |
|-----------|---------------------------|--------|
| 1 | $h(\text{initial})$ | DFS, élague tout $f >$ borne |
| 2 | min des $f$ élagués | DFS un peu plus profond |
| ... | ... | ... |
| k | $\geq f^*$ (solution) | Solution trouvée |

### Pourquoi IDA* ?

| Avantage | Détail |
|----------|--------|
| **Mémoire** | $O(bd)$ — linéaire en profondeur (vs $O(b^d)$ pour A*) |
| **Pas de file de priorité** | DFS récursif simple |
| **Même optimalité** | A* avec la même heuristique admissible |

**Inconvénient** : re-exploration des nœuds à chaque itération (comme IDDFS), mais l'overhead est modeste car la borne $f$ augmente peu à chaque cycle.

In [8]:
// IDA* : DFS recursif borne par f = g + h, borne croissante a chaque iteration.
SearchResult IDAStar(GraphProblem problem, Func<string, double> h, bool verbose = false) {
    var sw = Stopwatch.StartNew();
    double bound = h(problem.Initial);
    var root = new Node(problem.Initial);
    // Compteurs captures par la closure Dfs (mutables).
    int totalExpanded = 0;
    int totalGenerated = 0;
    var exploredOrder = new List<string>();

    // DFS recursif : retourne (solution, prochaine borne).
    (Node sol, double nextBound) Dfs(Node node, double curBound) {
        double f = node.PathCost + h(node.State);
        if (f > curBound) return (null, f);             // coupure
        if (problem.GoalTest(node.State)) return (node, curBound);
        totalExpanded++;
        exploredOrder.Add(node.State);
        double minNext = double.PositiveInfinity;
        foreach (var child in node.Expand(problem)) {
            totalGenerated++;
            var (sol, childBound) = Dfs(child, curBound);
            if (sol != null) return (sol, curBound);
            if (childBound < minNext) minNext = childBound;
        }
        return (null, minNext);
    }

    const int maxIters = 1000;
    int iterations = 0;
    while (iterations < maxIters) {
        iterations++;
        if (verbose) Console.WriteLine($"  Iteration {iterations}: f-limit = {bound:F0}");
        var (result, newBound) = Dfs(root, bound);
        if (result != null) {
            sw.Stop();
            return new SearchResult("IDA*", result, totalExpanded, totalGenerated, iterations, sw.Elapsed.TotalMilliseconds, exploredOrder);
        }
        if (double.IsInfinity(newBound)) break;  // espace epuise, pas de solution
        bound = newBound;
    }
    sw.Stop();
    return new SearchResult("IDA*", null, totalExpanded, totalGenerated, iterations, sw.Elapsed.TotalMilliseconds, exploredOrder);
}

Console.WriteLine("Algorithme IDA* implemente.");

Algorithme IDA* implemente.


In [9]:
// Test de IDA* sur Lille -> Toulouse : meme solution optimale qu'A*, mais memoire lineaire.
Console.WriteLine("IDA* : Lille -> Toulouse");
Console.WriteLine(new string('=', 60));
var resultIDA = IDAStar(problemLT, hLilleToulouse, verbose: true);
Console.WriteLine();
resultIDA.Display();

IDA* : Lille -> Toulouse


  Iteration 1: f-limit = 791


  Iteration 2: f-limit = 814


  Iteration 3: f-limit = 1021


  Iteration 4: f-limit = 1050


...

  Sauts  : 3


  Noeuds explores : 13


  Noeuds generes  : 38


  Pic frontiere   : 5


  Temps           : 1,57 ms


### Interprétation : IDA* et les bornes croissantes

**Sortie obtenue** : IDA* effectue plusieurs **DFS bornés** consécutifs. À chaque itération, la borne $f_{\text{limit}}$ augmente pour englober un peu plus de l'espace, jusqu'à atteindre le $f$ de la solution optimale.

| Itération | Borne $f_{\text{limit}}$ | Comportement |
|-----------|---------------------------|--------------|
| 1 | $h(\text{Lille}) \approx 830$ | DFS élague — pas encore de chemin sous cette borne |
| 2 | min des $f$ élagués | DFS un peu plus profond |
| ... | ... | converge vers $f^*$ |
| k | $\approx 1055 = f^*$ | Solution optimale trouvée |

**Points clés** :

1. IDA* trouve la **même solution optimale** qu'A* (1055 km) — l'optimalité est préservée.
2. **Mémoire linéaire** $O(bd)$ : IDA* ne stocke que la pile de récursion, pas de `PriorityQueue` expansive.
3. Le **coût** est la re-exploration : IDA* re-parcourt les nœuds peu profonds à chaque itération. Sur ce graphe, l'overhead est négligeable ; sur des espaces très profonds à faible ramification, il reste modéré.

> **Quand choisir IDA* ?** Quand l'espace de recherche est **profond**, la **mémoire est limitée**, et l'**heuristique discrétise bien** les $f$ (peu de valeurs distinctes — peu d'itérations).

---
## 6. Admissibilité et consistance : étude sur le 8-puzzle

### Définitions

| Terme | Définition |
|-------|------------|
| **Admissible** | $h(n) \leq h^*(n)$ pour tout $n$ — ne surestime jamais le coût optimal réel $h^*$. |
| **Consistante** (monotone) | $h(n) \leq c(n, a, n') + h(n')$ pour tout successeur $n'$ — l'estimation ne baisse pas plus vite que le coût du pas. |

**Relation** : consistante $\Rightarrow$ admissible. La réciproque est fausse.

### Deux heuristiques classiques du 8-puzzle

État : tableau 3×3 de 8 tuiles numérotées + 1 case vide (0). But : `(1,2,3,4,5,6,7,8,0)`.

| Heuristique | Définition | Admissible ? | Consistante ? |
|-------------|------------|--------------|---------------|
| **$h_1$ Tuiles mal placées** | Nombre de tuiles hors de leur position but | Oui | Pas toujours |
| **$h_2$ Distance Manhattan** | Somme des distances $L_1$ de chaque tuile à sa position but | Oui | **Oui** |

De plus, $h_2$ **domine** $h_1$ : $h_2(n) \geq h_1(n)$ partout — donc A* avec $h_2$ explore **au moins aussi peu** de nœuds qu'avec $h_1$.

In [10]:
// 8-Puzzle : etat = int[9], 0 = case vide. But = (1,2,3,4,5,6,7,8,0).
// (La tuile de valeur v a pour position but l'indice v-1 ; la case vide 0 va a l'indice 8.)

// h1 : nombre de tuiles mal placees (on exclut la case vide a l'indice 8).
int HMisplaced(int[] state) {
    int count = 0;
    for (int i = 0; i < 8; i++)       // indices 0..7 : tuiles 1..8 ; but[i] = i + 1
        if (state[i] != i + 1) count++;
    return count;
}

// h2 : somme des distances Manhattan de chaque tuile a sa position but.
int HManhattan(int[] state) {
    int dist = 0;
    for (int i = 0; i < 9; i++) {
        int v = state[i];
        if (v == 0) continue;            // la case vide ne compte pas
        int goalIdx = v - 1;             // la tuile v a pour position but l'indice v-1
        int curR = i / 3, curC = i % 3;
        int goalR = goalIdx / 3, goalC = goalIdx % 3;
        dist += Math.Abs(curR - goalR) + Math.Abs(curC - goalC);
    }
    return dist;
}

// Trois etats temoins (du plus facile au plus difficile).
int[] easy   = { 1, 2, 3, 4, 5, 0, 7, 8, 6 };    // 1 tuile decalee
int[] medium = { 1, 3, 4, 8, 0, 2, 7, 5, 6 };    // ~16 coups optimal
int[] hard   = { 8, 6, 7, 2, 5, 4, 3, 0, 1 };    // ~31 coups optimal

Console.WriteLine("Comparaison h1 (tuiles mal placees) vs h2 (Manhattan) sur 3 etats :");
Console.WriteLine(new string('-', 62));
Console.WriteLine($"{"Etat",-10} {"h1 (mal placees)",18} {"h2 (Manhattan)",18} {"h2 >= h1 ?",12}");
Console.WriteLine(new string('-', 62));
foreach (var (name, state) in new[] { ("Facile", easy), ("Moyen", medium), ("Difficile", hard) }) {
    int h1 = HMisplaced(state);
    int h2 = HManhattan(state);
    Console.WriteLine($"{name,-10} {h1,18} {h2,18} {(h2 >= h1 ? "oui" : "NON"),12}");
}
Console.WriteLine();
Console.WriteLine("Conclusion : h2 (Manhattan) domine h1 partout -> A* avec h2 explorerait");
Console.WriteLine("au moins aussi peu de noeuds qu'avec h1. Les deux sont admissibles.");

Comparaison h1 (tuiles mal placees) vs h2 (Manhattan) sur 3 etats :


--------------------------------------------------------------


Etat         h1 (mal placees)     h2 (Manhattan)   h2 >= h1 ?


--------------------------------------------------------------


Facile                      1                  1          oui


Moyen                       6                 10          oui


Difficile                   7                 21          oui


Conclusion : h2 (Manhattan) domine h1 partout -> A* avec h2 explorerait


au moins aussi peu de noeuds qu'avec h1. Les deux sont admissibles.


In [11]:
// Demonstration de consistance : pour Manhattan, h(n) <= c(n,n') + h(n') pour toute transition.
// On prend un etat, on genere un successeur (deplacement de la case vide), on verifie.
string StateStr(int[] s) => "(" + string.Join(",", s) + ")";

// Genere les successeurs d'un etat du 8-puzzle (deplacement de la case vide).
IEnumerable<int[]> PuzzleSuccessors(int[] state) {
    int blank = Array.IndexOf(state, 0);
    int r = blank / 3, c = blank % 3;
    int[][] moves = { new[] { -1, 0 }, new[] { 1, 0 }, new[] { 0, -1 }, new[] { 0, 1 } };
    foreach (var m in moves) {
        int nr = r + m[0], nc = c + m[1];
        if (nr < 0 || nr >= 3 || nc < 0 || nc >= 3) continue;
        int ni = nr * 3 + nc;
        var copy = (int[])state.Clone();
        (copy[blank], copy[ni]) = (copy[ni], copy[blank]);
        yield return copy;
    }
}

var testState = medium;   // etat Moyen
int hParent = HManhattan(testState);
Console.WriteLine($"Demonstration de consistance de Manhattan sur un etat Moyen :");
Console.WriteLine($"  Etat parent : {StateStr(testState)}");
Console.WriteLine($"  h2(parent)  = {hParent}");
Console.WriteLine();
Console.WriteLine($"  {"Successeur",-22} {"h2(succ)",10} {"h(parent)-h(succ)",20} {"<= c=1 ?",10}");
Console.WriteLine($"  {new string('-', 62)}");
foreach (var succ in PuzzleSuccessors(testState)) {
    int hSucc = HManhattan(succ);
    int diff = hParent - hSucc;
    Console.WriteLine($"  {StateStr(succ),-22} {hSucc,10} {diff,20} {(diff <= 1 ? "oui" : "NON"),10}");
}
Console.WriteLine();
Console.WriteLine("On observe h(parent) - h(succ) <= 1 a chaque transition : un deplacement de la");
Console.WriteLine("case vide ne rapproche une tuile de sa but que d'au plus 1 (cout unitaire).");
Console.WriteLine("Donc h(n) <= 1 + h(n') : Manhattan est CONSISTANTE (et donc admissible).");

Demonstration de consistance de Manhattan sur un etat Moyen :


  Etat parent : (1,3,4,8,0,2,7,5,6)


  h2(parent)  = 10


  Successeur               h2(succ)    h(parent)-h(succ)   <= c=1 ?


  --------------------------------------------------------------


...

  (1,3,4,0,8,2,7,5,6)             9                    1        oui


  (1,3,4,8,2,0,7,5,6)             9                    1        oui


On observe h(parent) - h(succ) <= 1 a chaque transition : un deplacement de la


case vide ne rapproche une tuile de sa but que d'au plus 1 (cout unitaire).


Donc h(n) <= 1 + h(n') : Manhattan est CONSISTANTE (et donc admissible).


### Interprétation : choix d'une bonne heuristique

**Sortie obtenue** : sur les trois états témoins, $h_2$ (Manhattan) est toujours $\geq h_1$ (tuiles mal placées) — la **dominance** est vérifiée. La table de consistance montre qu'à chaque déplacement de la case vide, $h_2$ ne baisse **jamais de plus de 1** (le coût du pas) : c'est précisément la définition de la **consistance**.

| Propriété | $h_1$ Tuiles mal placées | $h_2$ Distance Manhattan |
|-----------|--------------------------|---------------------------|
| Admissible | Oui | Oui |
| Consistante | Pas toujours | **Oui** |
| Dominance | dominée par $h_2$ | **domine** $h_1$ |
| Effet sur A* | plus de nœuds explorés | **moins** de nœuds explorés |

**Leçon pratique** : pour A* sur le 8-puzzle, **Manhattan est strictement supérieure** — à la fois consistante (pas de re-open de nœuds) et dominante (moins de nœuds explorés). C'est l'heuristique canonique utilisée dans tous les solveurs de taquin.

---
## 7. Exercices

Les exercices sont à compléter : les squelettes s'exécutent sans erreur (ils retournent un résultat neutre) — à vous d'implémenter la logique demandée. Conservez les commentaires `// Indice` et `// Étape N`.

### Exercice 1 : DFS limité par $f$ (brique de base d'IDA*)

**Énoncé** : implémentez le DFS récursif borné par $f(n) = g(n) + h(n)$ qui est le cœur d'IDA*. La signature et le squelette sont fournis. Testez en reconstruisant IDA* à partir de votre brique.

In [12]:
// Exercice 1 : DFS recursif limite par f = g + h (brique de base d'IDA*).
// TODO etudiant : implementer la coupure sur f > bound et la propagation du nextBound.
(Node sol, double nextBound) DfsLimited(Node node, double bound, GraphProblem problem, Func<string, double> h) {
    // Indice : f = node.PathCost + h(node.State).
    //   - si f > bound : retourner (null, f) -> coupure, f remonte comme prochain candidate.
    //   - si problem.GoalTest(node.State) : retourner (node, bound) -> solution trouvee.
    //   - sinon : explorer recursivement chaque fils via node.Expand(problem),
    //     garder le MIN des nextBound retournes par les fils coupes.
    // Etape 1 : calculer f.
    // Etape 2 : tester le but.
    // Etape 3 : boucler sur les fils, retourner la premiere solution, sinon le min des nextBound.
    return (null, double.PositiveInfinity);  // TODO etudiant : remplacer par la vraie logique
}

// Test (placeholder neutre — doit s'executer sans erreur)
var pb1 = new GraphProblem("Lille", "Toulouse", franceGraph);
var (s1, b1) = DfsLimited(new Node("Lille"), hLilleToulouse("Lille"), pb1, hLilleToulouse);
Console.WriteLine($"(Exercice a completer) — nextBound retourne par le stub : {b1}");

(Exercice a completer) — nextBound retourne par le stub : ∞


### Exercice 2 : Heuristique volontairement non admissible

**Énoncé** : construisez une heuristique $h_{\text{bad}}(n) = 2 \cdot \text{Haversine}(n, \text{but})$ — qui **sur-estime** donc n'est **pas admissible**. Lancez A* avec $h_{\text{bad}}$ sur Lille → Toulouse et comparez :

1. Le coût trouvé (est-il encore optimal ?)
2. Le nombre de nœuds explorés (plus ou moins qu'avec $h$ admissible ?)

**Indice** : une heuristique non admissible « ferme » trop tôt des chemins prometteurs (elle les juge trop chers), ce qui peut accélérer la recherche mais au prix de l'optimalité.

In [13]:
// Exercice 2 : Heuristique volontairement NON admissible (x2) et comparaison avec A*.
// TODO etudiant : construire h_bad(s) = 2 * Haversine(s, goal), lancer AStar, comparer.
Func<string, double> MakeNonAdmissibleH(string goal) {
    // Indice : return s => 2.0 * Haversine(s, goal);
    // Etape 1 : definir la lambda ci-dessus (sur-estime par 2 -> non admissible).
    // Etape 2 : appeler AStar(pb, h_bad) sur Lille -> Toulouse.
    // Etape 3 : comparer le cout trouve au cout optimal ~1055 km et le nombre de noeuds.
    return null;  // TODO etudiant : remplacer par la lambda
}

var hBad = MakeNonAdmissibleH("Toulouse");
if (hBad == null) {
    Console.WriteLine("(Exercice a completer) — heuristique non admissible a construire");
} else {
    var rBad = AStar(problemLT, hBad);
    Console.WriteLine($"A* avec h_bad (x2) : cout = {rBad.Cost:F0} km, noeuds = {rBad.NodesExpanded}");
    Console.WriteLine($"A* avec h admissible : cout = {resultAStar.Cost:F0} km, noeuds = {resultAStar.NodesExpanded}");
}

(Exercice a completer) — heuristique non admissible a construire


### Exercice 3 : Pathfinding A* sur une grille pondérée 10×10

**Énoncé** : adaptez A* à un problème de **pathfinding sur grille 10×10** où chaque cellule a un **coût de terrain** différent (1 = route rapide, 3 = herbe, 5 = boue, 0 = obstacle). Comparez deux heuristiques :

1. **Manhattan pure** : $h(n) = |\Delta r| + |\Delta c|$ — admissible mais peu informative.
2. **Manhattan pondérée** : $h_w(n) = c_{\min} \cdot (|\Delta r| + |\Delta c|)$ où $c_{\min}$ = coût minimum de terrain (ici 1). **Justifiez** qu'elle est admissible et **domine** la Manhattan pure.

**Indice** : représentez la grille comme `int[10,10]`, l'état comme un `(int r, int c)`. Définissez un `GridProblem` sous-classe de `Problem` (avec `Actions`/`Result`/`StepCost`) puis adaptez A* pour comparer le nombre de nœuds explorés.

In [14]:
// Exercice 3 : Pathfinding A* sur grille ponderee 10x10 (couts de terrain variables).
// TODO etudiant : definir la grille, l'heuristique de Manhattan ponderee, lancer A*.
void GrillePondereeAStar() {
    // Indice : grille 10x10, 0 = obstacle, k >= 1 = cout de traversee.
    //   1 = route rapide, 3 = herbe, 5 = boue.
    // Etape 1 : construire int[10,10] terrain avec un melange de zones.
    // Etape 2 : definir un GridProblem (sous-classe de Problem) :
    //             - Actions(state) : 4 directions cardinales valides (hors obstacle).
    //             - Result(state, action) : (r+dr, c+dc).
    //             - StepCost(...) : terrain[nr, nc].
    // Etape 3 : choisir l'heuristique admissible = Manhattan * c_min (c_min = 1 ici).
    // Etape 4 : adapter AStar pour ce probleme et afficher le chemin trouve + nb de noeuds.
    // Etape 5 : comparer avec Manhattan pure — meme cout optimal, mais noeuds explores differents.
    Console.WriteLine("(Exercice a completer) — grille ponderee 10x10 a implementer");
}

GrillePondereeAStar();

(Exercice a completer) — grille ponderee 10x10 a implementer


---
## 8. Conclusion

### Concepts clés

| Concept | Définition |
|---------|------------|
| **Heuristique $h(n)$** | Estimation du coût restant de $n$ au but |
| **Admissible** | $h(n) \leq h^*(n)$ (ne surestime jamais) |
| **Consistante** | $h(n) \leq c(n, n') + h(n')$ (monotone ; implique admissible) |
| **Dominance** | $h_1 \geq h_2$ partout : $h_1$ est plus informée |
| **$f(n) = g(n) + h(n)$** | Fonction d'évaluation de A* |

### Algorithmes

| Algorithme | Critère | Optimal | Mémoire | Quand ? |
|------------|---------|---------|---------|---------|
| **Greedy** | minimiser $h(n)$ | Non | $O(b^m)$ | Vitesse prioritaire, solution « assez bonne » |
| **A\*** | minimiser $g(n) + h(n)$ | **Oui** (h admissible) | $O(b^d)$ | Référence, optimalité requise |
| **IDA\*** | DFS itéré par $f(n)$ | **Oui** | $O(bd)$ | Mémoire limitée, espace profond |

### Ce qu'il faut retenir

1. **Greedy vs A* sur Lille → Toulouse** : Greedy (1210 km) est ~14% sous-optimal vs A* (1055 km) — la divergence est **structurelle**, pas accidentelle.
2. **A* est l'algorithme de référence** : optimal avec heuristique admissible, c'est le choix par défaut.
3. **IDA* pour la mémoire** : même optimalité qu'A*, consommation **linéaire** au lieu d'exponentielle.
4. **La qualité de l'heuristique est déterminante** : Manhattan (consistante, dominante) explore bien moins de nœuds que « tuiles mal placées » sur le 8-puzzle.

### Références

- Hart, P. E., Nilsson, N. J. & Raphael, B. (1968). *A Formal Basis for the Heuristic Determination of Minimum Cost Paths*. IEEE Trans. SSC 4(2):100-107.
- Korf, R. E. (1985). *Depth-first Iterative-Deepening: An Optimal Admissible Tree Search*. Artificial Intelligence 27(1):97-109.
- Russell, S. & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.), Pearson — ch. 3.5.

---

**Navigation** : [<< Recherche non informée (Search-2 C#)](Search-2-Uninformed-Csharp.ipynb) | [Index série Search](../README.md) | [Recherche locale (Search-4 C#) >>](Search-4-LocalSearch-Csharp.ipynb)

*Port C# / .NET 9.0 du notebook Python Search-3-Informed — Epic [#4956](https://github.com/jsboige/CoursIA/issues/4956) (parité .NET ⇄ Python des séries).*